In [ ]:
# If needed: !pip install -U ultralytics==8.3.50  # or newer
from ultralytics import YOLO
import sys, os, shutil, random
from pathlib import Path
import yaml
import glob

print("Python:", sys.version)
print("Ultralytics:", YOLO.__module__)


Python: 3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]
Ultralytics: ultralytics.models.yolo.model


In [ ]:
# Set your raw dataset layout here. Common layout:
#   dataset_root/
#       images/            <-- all images in one folder (or nested)
#       labels/            <-- YOLO txt labels, same relative structure and stems
#
# The splitter will crawl images/** and only include those that have labels/**.txt

dataset_root = Path("datasets/raw_detection").resolve()

# Where to create the split dataset (symlinks or copies into YOLO structure)
out_root = Path("datasets/detection_partial").resolve()

# Image and label subfolders relative to dataset_root
images_dirname = "images"
labels_dirname = "labels"

# Supported image extensions
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Split ratios
train_ratio = 0.80
val_ratio   = 0.20
test_ratio  = 0

# If True use symlinks to save space, else copy files
use_symlinks = True

# Class names in order of class id in your labels
# Replace with your actual names
# Class names in order of class id in your labels
class_names = ["blue", "yellow", "red", "brown", "subject"]

# Base YOLOv11 model to start from - pick one: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt
base_model = "yolo11n.pt"

# Training hyperparameters
epochs = 100
imgsz  = 640
batch  = 16   # or an int
device = 'cuda'        # set to "cpu" to force CPU, or 0/1/... for GPU

dataset_root, out_root


(PosixPath('/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection'),
 PosixPath('/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/detection_partial'))

In [3]:
def find_all_images(root: Path, images_dirname: str, img_exts: set):
    images_root = root / images_dirname
    if not images_root.exists():
        raise FileNotFoundError(f"Images folder not found: {images_root}")
    # recursive glob for all supported extensions
    files = []
    for ext in img_exts:
        files.extend(images_root.rglob(f"*{ext}"))
    return [p for p in files if p.is_file()]

def image_to_label_path(img_path: Path, dataset_root: Path, images_dirname: str, labels_dirname: str):
    # Convert images/.../name.jpg -> labels/.../name.txt with same relative subpath
    rel = img_path.relative_to(dataset_root / images_dirname)
    return dataset_root / labels_dirname / rel.with_suffix(".txt")

all_images = find_all_images(dataset_root, images_dirname, img_exts)
print(f"Found {len(all_images)} candidate images.")

# Keep only images that have a corresponding label file with at least one line (optional check)
valid_pairs = []
missing = []
empty = []

for img in all_images:
    lb = image_to_label_path(img, dataset_root, images_dirname, labels_dirname)
    if lb.exists():
        # Optional: require non-empty labels
        text = lb.read_text(encoding="utf-8").strip()
        if text:
            valid_pairs.append((img, lb))
        else:
            empty.append((img, lb))
    else:
        missing.append(img)

print(f"Usable pairs: {len(valid_pairs)}")
print(f"Missing label files: {len(missing)}")
print(f"Empty label files: {len(empty)}")


Found 634 candidate images.
Usable pairs: 330
Missing label files: 304
Empty label files: 0


In [ ]:
random_seed = 2025
random.Random(random_seed).shuffle(valid_pairs)

n = len(valid_pairs)
n_train = int(n * train_ratio)
n_val   = int(n * val_ratio)
n_test  = n - n_train - n_val

train_pairs = valid_pairs[:n_train]
val_pairs   = valid_pairs[n_train:n_train+n_val]
test_pairs  = valid_pairs[n_train+n_val:]

print(f"Train: {len(train_pairs)}")
print(f"Val:   {len(val_pairs)}")
print(f"Test:  {len(test_pairs)}")
assert len(train_pairs) + len(val_pairs) + len(test_pairs) == n


Train: 264
Val:   66
Test:  0


In [5]:
def safe_make_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def place_pair(img: Path, lb: Path, dest_img: Path, dest_lb: Path, symlink=True):
    safe_make_dir(dest_img.parent)
    safe_make_dir(dest_lb.parent)
    if symlink:
        # remove existing if any
        if dest_img.exists() or dest_img.is_symlink():
            dest_img.unlink()
        if dest_lb.exists() or dest_lb.is_symlink():
            dest_lb.unlink()
        os.symlink(img, dest_img)
        os.symlink(lb, dest_lb)
    else:
        if not dest_img.exists():
            shutil.copy2(img, dest_img)
        if not dest_lb.exists():
            shutil.copy2(lb, dest_lb)

def materialize_split(pairs, out_split_dir: Path, images_dirname="images", labels_dirname="labels", symlink=True):
    for img, lb in pairs:
        # keep relative structure under images_dirname/ and labels_dirname/
        rel_img = img.relative_to(dataset_root / images_dirname)
        dest_img = out_split_dir / images_dirname / rel_img
        rel_lb  = lb.relative_to(dataset_root / labels_dirname)
        dest_lb = out_split_dir / labels_dirname / rel_lb
        place_pair(img, lb, dest_img, dest_lb, symlink=symlink)

# YOLO structure: out_root/{train,val,test}/{images,labels}
for split in ["train", "val", "test"]:
    safe_make_dir(out_root / split / "images")
    safe_make_dir(out_root / split / "labels")

materialize_split(train_pairs, out_root / "train", images_dirname, labels_dirname, use_symlinks)
materialize_split(val_pairs,   out_root / "val",   images_dirname, labels_dirname, use_symlinks)
materialize_split(test_pairs,  out_root / "test",  images_dirname, labels_dirname, use_symlinks)

print("Split materialized at:", out_root)


Split materialized at: /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/detection_partial


In [ ]:
def count_files(p: Path, pattern="*"):
    return len(list(p.rglob(pattern)))

for split in ["train", "val", "test"]:
    ni = count_files(out_root / split / "images")
    nl = count_files(out_root / split / "labels", "*.txt")
    print(f"{split:>5} - images: {ni:6d}  labels: {nl:6d}")

# Show a few matched paths to confirm structure visually
for i, (img, lb) in enumerate(train_pairs[:5]):
    print(str(img), "->", str(lb))


train - images:    264  labels:    264
  val - images:     66  labels:     66
 test - images:      0  labels:      0
/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/images/S__2383912_rect_R1C3.jpg -> /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/labels/S__2383912_rect_R1C3.txt
/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/images/S__2383878_rect_R2C1.jpg -> /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/labels/S__2383878_rect_R2C1.txt
/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/images/S__2383902_rect_R2C1.jpg -> /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/labels/S__2383902_rect_R2C1.txt
/media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/images/S__2375694_rect_R2C1.jpg -> /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/raw_detection/labels/S__2375694_rect_R2C1.txt
/media/tameszaza/MyPassport/ml/OPH_project/2025

In [ ]:
data_yaml_path = out_root / "dataset.yaml"
data_dict = {
    "path": str(out_root),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {i: name for i, name in enumerate(class_names)}
}
with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_dict, f, sort_keys=False, allow_unicode=True)

print(data_yaml_path.read_text())


path: /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/detection_partial
train: train/images
val: val/images
test: test/images
names:
  0: blue
  1: yellow
  2: red
  3: brown
  4: subject



In [ ]:
# Load model and train
model = YOLO(base_model)  # example: yolo11n.pt
results = model.train(
    data=str(data_yaml_path),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    device=device,
    project=str(out_root / "runs"),
    name="yolo11_det",
    exist_ok=True,
    verbose=True,
)
results


New https://pypi.org/project/ultralytics/8.3.176 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.50 🚀 Python-3.12.3 torch-2.3.1+cu121 


/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:118: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 1
os.environ['CUDA_VISIBLE_DEVICES']: None
